In [ ]:
import requests
import json

In [ ]:
# Configuration
DATACENTER_ID = ""
API_TOKEN = ""
BASE_URL = f"https://{DATACENTER_ID}.qualtrics.com/API/v3/"
HEADERS = {
    "Content-Type": "application/json",
    "X-API-TOKEN": API_TOKEN
}

In [ ]:
def test_api_connection():
    """Test Qualtrics API connection"""
    try:
        response = requests.get(BASE_URL + "whoami", headers=HEADERS)
        response.raise_for_status()
        user_info = response.json()
        print("✓ API Connection Successful")
        print(f"User Info: {json.dumps(user_info, indent=2)}")
        return True
    except requests.exceptions.RequestException as e:
        print(f"✗ Connection Failed: {str(e)}")
        if response:
            print(f"Status Code: {response.status_code}")
            print(f"Response: {response.text}")
        return False

In [ ]:
def create_survey():
    """Create a new survey with sample questions"""
    survey_payload = {
        "SurveyName": "Bocconi Feedback Survey",
        "Language": "EN",
        "ProjectCategory": "CORE",
    }


    try:
        response = requests.post(
            BASE_URL + "survey-definitions",
            headers=HEADERS,
            json=survey_payload
        )
        response.raise_for_status()

        result = response.json()
        if result["meta"]["httpStatus"] == "200 - OK":
            survey_id = result["result"]["SurveyID"]
            print(f"✓ Survey created successfully!\nSurvey ID: {survey_id}")
            # Add a yes/no question
            add_question(survey_id)
            return survey_id
        else:
            print("✗ Survey creation failed")
            print(json.dumps(result, indent=2))
            return None

    except requests.exceptions.RequestException as e:
        print(f"✗ API Request Failed: {str(e)}")
        if response:
            print(f"Status Code: {response.status_code}")
            print(f"Response: {response.text}")
        return None

In [ ]:
def add_question(survey_id):
    """Adds questions to the survey based on user input."""

    question_types = ["text", "mc", "matrix"]  # Supported question types
    num_questions = {}

    # Get the number of questions for each type from the user
    for question_type in question_types:
        while True:
            try:
                num = int(input(f"How many {question_type} questions do you want to add? "))
                if num >= 0:
                    num_questions[question_type] = num
                    break
                else:
                    print("Please enter a non-negative number.")
            except ValueError:
                print("Invalid input. Please enter a number.")

    # Add questions for each type
    for question_type in question_types:
        for _ in range(num_questions[question_type]):
            print(f"\nAdding a {question_type} question:")
            question_payload = {}

            # Handle text questions
            if question_type == "text":
                question_payload = {
                    "QuestionText": input("Enter the QuestionText: "),
                    "DataExportTag": "Q4",
                    "QuestionType": "TE",
                    "Selector": "SL",
                    "Configuration": {
                        "QuestionDescriptionOption": "UseText"
                    },
                    "QuestionDescription": "Please provide your city of residence."
                }

            # Handle mc questions
            elif question_type == "mc":
                question_payload = {
                    "QuestionText": input("Enter the QuestionText: "),
                    "DataExportTag": "Q2",
                    "QuestionType": "MC",
                    "Selector": "SAVR",
                    "SubSelector": "TX",
                    "Configuration": {
                        "QuestionDescriptionOption": "UseText"
                    },
                }
                choices = {}
                num_choices = int(input("Enter the number of choices: "))
                for i in range(1, num_choices + 1):
                    choices[str(i)] = {"Display": input(f"Enter choice {i}: ")}
                question_payload["Choices"] = choices
                question_payload["ChoiceOrder"] = list(choices.keys())

            # Handle matrix questions
            elif question_type == "matrix":
                question_payload = {
                    "QuestionText": input("Enter the QuestionText: "),
                    "DataExportTag": "Q3",
                    "QuestionType": "Matrix",
                    "Selector": "Likert",
                    "SubSelector": "SingleAnswer",
                    "Configuration": {
                        "QuestionDescriptionOption": "UseText"
                    },
                }
                choices = {}
                num_choices = int(input("Enter the number of choices: "))
                for i in range(1, num_choices + 1):
                    choices[str(i)] = {"Display": input(f"Enter choice {i}: ")}
                question_payload["Choices"] = choices
                question_payload["ChoiceOrder"] = list(choices.keys())
                answers = {}
                num_answers = int(input("Enter the number of answers: "))
                for i in range(1, num_answers + 1):
                    answers[str(i)] = {"Display": input(f"Enter answer {i}: ")}
                question_payload["Answers"] = answers
                question_payload["AnswerOrder"] = list(answers.keys())

            # Send the API request to add the question
            try:
                response = requests.post(
                    f"{BASE_URL}survey-definitions/{survey_id}/questions",
                    headers=HEADERS,
                    json=question_payload
                )
                response.raise_for_status()
                print("✓ Question added successfully!")
            except requests.exceptions.RequestException as e:
                print(f"✗ API Request Failed: {str(e)}")

In [ ]:
def get_survey_preview_link(survey_id):
    response = requests.get(f"{BASE_URL}/surveys/{survey_id}", headers=HEADERS)
    if response.status_code == 200:
        # This API call gives survey info, but doesn't have a direct preview link
        # The preview link follows a known pattern:
        preview_link = f"https://{DATACENTER_ID}.qualtrics.com/jfe/preview/{survey_id}"
        print(f"Preview link: {preview_link}")
        return preview_link
    else:
        print("Error retrieving survey info:", response.status_code, response.text)
        return None

In [ ]:
# Execute the workflow
if __name__ == "__main__":
    if test_api_connection():
        print("\nCreating survey...")
        survey_id = create_survey()
    if survey_id:
        get_survey_preview_link(survey_id)

In [ ]:
#https://YOUR_DATA_CENTER.qualtrics.com/jfe/preview/SURVEY_ID
#Survey ID: SV_8pGdjY0sAWGzZGu
#Preview link: https://fra1.qualtrics.com/jfe/preview/SV_8pGdjY0sAWGzZGu
survey_id

In [ ]:
#it creates but not viewable or published.

In [ ]:
# Step 1: Publish the survey
publish_url = publish_link = get_survey_preview_link(survey_id)
publish_response = requests.post(publish_url, headers=HEADERS)

In [ ]:
if publish_response.status_code == 200:
    print("Survey published successfully!")
else:
    print("Failed to publish survey.")
    print(f"Response: {publish_response.text}")

In [ ]:
publish_response

In [ ]:
survey_id

In [ ]:
#activate survey
activation_url = f"https://bocconi.fra1.qualtrics.com/API/v3/surveys/{survey_id}/activate"
activation_payload = {
    "isActive": True
}
activation_response = requests.put(activation_url, headers=HEADERS, data=json.dumps(activation_payload))
activation_response

In [ ]:
BASE_URL = f'https://bocconi.eu.qualtrics.com/API/v3/surveys/'
url = f"{BASE_URL}{survey_id}"
data = {
    "isActive": True
}
response = requests.put(url, headers=HEADERS, data=json.dumps(data))
print(response.status_code)
print(response.json())

In [ ]:
response = requests.put(url, headers=HEADERS, json=data)
print(response.status_code)
print(response.json())

In [ ]:
survey_id

In [ ]:
#it should actually be -
#https://bocconi.eu.qualtrics.com/jfe/form/SV_cZrWXXTZCFc8lo2
#rewrite the code in that form
print(f"https://bocconi.eu.qualtrics.com/jfe/form/{survey_id}")